# ICS ERC test for Qwen 9B

This notebook runs the ERC-balanced ICS path on a Qwen 9B target in Colab.

Model note: `Qwen/Qwen3.6-9B` was not found as an official Qwen repo when this notebook was created. The default below is the official base checkpoint `Qwen/Qwen3.5-9B-Base`, not an instruct/chat derivative. If you meant a specific community `Qwen3.6-9B` base repo, change `MODEL_ID` in the configuration cell.

In [ ]:
# Runtime setup. Use a high-RAM GPU runtime for 9B.
!nvidia-smi
!python -V
!pip install -q --upgrade pip
!pip install -q torch transformers accelerate bitsandbytes safetensors scipy huggingface_hub

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/toxzak-svg/ICS.git"
BRANCH = "codex-qwen3-06b-pipeline-docs"

MODEL_ID = "Qwen/Qwen3.5-9B-Base"
OUTPUT_DIR = Path("/content/qwen9b-ics-erc")
RESULTS_JSON = Path("/content/qwen9b-erc-results.json")
LOG_PATH = Path("/content/qwen9b-erc.log")

MAX_CALIBRATION_SAMPLES = 16
MAX_CALIBRATION_LENGTH = 192
MAX_CHAINS = None
ERC_MAX_RELATIVE_ERROR = 0.10

UPLOAD_TO_HF = False
HF_MODEL_REPO = "toxzak/Qwen3.5-9B-Base-ICS-ERC"
HF_DATASET_REPO = "toxzak/ics-quantization-artifacts"


In [ ]:
if Path("/content/ICS").exists():
    %cd /content/ICS
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/ICS
    %cd /content/ICS

!git rev-parse --short HEAD
!pip install -q -e .

In [ ]:
import json
import math
import time
import traceback
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from ics.pipeline import ICSConfig, quantize_model
from ics.export import save_ics_model, load_ics_model, dequantized_state_dict
from scripts.colab_quantize_qwen35 import DEFAULT_CALIBRATION

torch.manual_seed(42)
LOG_PATH.write_text("")

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

assert torch.cuda.is_available(), "This notebook needs a CUDA Colab runtime."
log(f"GPU: {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info()
log(f"VRAM free/total: {free/1e9:.2f} / {total/1e9:.2f} GB")

In [ ]:
log(f"Loading {MODEL_ID} in 4-bit NF4")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    trust_remote_code=True,
)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

log("Model loaded")

In [ ]:
config = ICSConfig(
    block_size=64,
    int4_fraction=0.5,
    int2_fraction=0.4,
    int1_fraction=0.1,
    method="composite",
    calibration_texts=DEFAULT_CALIBRATION[:MAX_CALIBRATION_SAMPLES],
    max_calibration_length=MAX_CALIBRATION_LENGTH,
    max_chains=MAX_CHAINS,
    fisher_loss_mode="last_logit_mean",
    erc_enabled=True,
    erc_max_relative_error=ERC_MAX_RELATIVE_ERROR,
)

log("Starting ERC-balanced ICS quantization")
t0 = time.time()
result = quantize_model(model, tokenizer, config, device="cuda", show_progress=True)
elapsed = time.time() - t0
log(f"Quantization complete in {elapsed/60:.1f} min; quantized_layers={len(result.quant)} perms={len(result.perms)}")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_ics_model(result, OUTPUT_DIR, tokenizer=tokenizer)

promoted = 0
blocks = 0
for qt in result.quant.values():
    if qt.erc_promoted is not None:
        promoted += int(qt.erc_promoted.sum().item())
        blocks += int(qt.erc_promoted.numel())

artifact_bytes = sum(p.stat().st_size for p in OUTPUT_DIR.rglob("*") if p.is_file())
summary = {
    "model_id": MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "artifact_bytes": artifact_bytes,
    "quantized_layers": len(result.quant),
    "permutations": len(result.perms),
    "erc_max_relative_error": ERC_MAX_RELATIVE_ERROR,
    "erc_promoted_blocks": promoted,
    "erc_total_blocks": blocks,
    "elapsed_seconds": elapsed,
}
RESULTS_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

In [ ]:
# Short dense-load validation. Keep token count small for 9B.
EVAL_TEXT = "Transformers route information through attention and feed-forward layers. Quantization changes storage precision but should preserve useful logits."

@torch.no_grad()
def ppl_for_model(m, tok, text, max_tokens=96):
    ids = tok(text, return_tensors="pt", truncation=False).input_ids[:, :max_tokens].to("cuda")
    out = m(input_ids=ids, use_cache=False)
    logits = out.logits[:, :-1, :].float()
    labels = ids[:, 1:]
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), reduction="mean")
    return float(torch.exp(loss).item()), int(labels.numel())

base_ppl, tokens = ppl_for_model(model, tokenizer, EVAL_TEXT)
loaded = load_ics_model(OUTPUT_DIR)
state = dequantized_state_dict(loaded)
missing, unexpected = model.load_state_dict(state, strict=False)
ics_ppl, _ = ppl_for_model(model, tokenizer, EVAL_TEXT)

summary.update({
    "validation_tokens": tokens,
    "base_4bit_loaded_ppl": base_ppl,
    "ics_dequantized_ppl": ics_ppl,
    "dequantized_tensors_loaded": len(state),
    "base_tensors_unchanged": len(missing),
    "unexpected_state_keys": unexpected,
})
RESULTS_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

In [ ]:
# Optional upload. Set UPLOAD_TO_HF=True and provide HF_TOKEN in Colab secrets/env.
if UPLOAD_TO_HF:
    from huggingface_hub import HfApi
    token = os.environ.get("HF_TOKEN")
    assert token, "Set HF_TOKEN before uploading."
    api = HfApi(token=token)
    api.create_repo(HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True)
    api.create_repo(HF_DATASET_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(OUTPUT_DIR), commit_message="Add Qwen 9B ERC ICS artifact")
    api.upload_file(repo_id=HF_DATASET_REPO, repo_type="dataset", path_or_fileobj=str(RESULTS_JSON), path_in_repo=RESULTS_JSON.name, commit_message="Add Qwen 9B ERC test results")
    api.upload_file(repo_id=HF_DATASET_REPO, repo_type="dataset", path_or_fileobj=str(LOG_PATH), path_in_repo=LOG_PATH.name, commit_message="Add Qwen 9B ERC log")
    print("Uploaded", HF_MODEL_REPO, HF_DATASET_REPO)
else:
    print("UPLOAD_TO_HF is False; artifacts remain local in this Colab runtime.")